In [1]:
import sys
sys.path.append('/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages')

In [ ]:
print('This is Maggie's notebook)

In [2]:
import numpy as np
import sounddevice as sd 
import lightkurve as lk
import matplotlib.pyplot as plt 

ModuleNotFoundError: No module named 'lightkurve'

In [3]:
tic = 124029677 
sector = 33

search_result = lk.search_lightcurve(
        f"TIC {tic}",
        mission="TESS",
        author="SPOC",
        sector =[sector]
    )

lc = search_result.download()


NameError: name 'lk' is not defined

In [ ]:
lc_binned = lc.bin(time_bin_size=0.01)

In [ ]:
lc_fluxes = lc_binned.flux.value
lc_times = lc_binned.time.value


In [ ]:
plt.scatter(lc_times, lc_fluxes)

In [ ]:
len(lc_times)

In [ ]:
med_value = np.nanmedian(lc_fluxes)
lc_normflux = lc_fluxes/med_value

#central_val = 440 # hz 
max_val =900
min_val = 250

mapped_flux = (lc_normflux - np.nanmin(lc_normflux)) / (np.nanmax(lc_normflux) - np.nanmin(lc_normflux)) * (max_val - min_val) + min_val

plt.scatter(lc_times[1580:1650], mapped_flux[1580:1650])
# plt.scatter(lc_times, mapped_flux)

In [ ]:
lc_normflux

In [ ]:
# for a loop
freq_maps = ()

samplerate = 44100
duration = 0.05

for flux in mapped_flux[1580:1650]:
    frequency = flux 
    t = np.linspace(0, duration, int(samplerate * duration), endpoint=False)
    
    audio_signal = 0.5 * np.sin(2*np.pi*frequency*t)

    sd.play(audio_signal, samplerate)
    sd.wait()


In [ ]:
from scipy.io.wavfile import write

In [ ]:
samplerate = 44100
duration = 0.05

audio = []

for flux in mapped_flux[1580:1650]:
    frequency = flux 
    t = np.linspace(0, duration, int(samplerate * duration), endpoint=False)
    
    audio_signal = np.sin(2*np.pi*frequency*t)
    
    audio.extend(audio_signal)

arr = np.array(audio)
    

In [ ]:
sd.play(arr, samplerate)
sd.wait()

In [ ]:
import matplotlib.animation as animation

In [ ]:
plt.scatter(x,y)

In [ ]:
len(x)

In [ ]:
fig = plt.figure()
x = lc_times[1580:1650]
y = mapped_flux[1580:1650]

graph, = plt.plot([], [], 'o')

def animate(i):
    graph.set_data(x[:i+1], y[:i+1])
    return graph


# def animate(i):
#     scat.set_offsets((lc_times[1580:1650]))
#     return (scat,)

ani = animation.FuncAnimation(fig, animate, frames=len(x))
plt.show()


In [ ]:
# Copy
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

x = np.arange(10)
y = np.random.random(10)
size = np.random.randint(150, size=10)
colors = np.random.choice(["r", "g", "b"], size=10)

fig = plt.figure()
plt.xlim(0, 10)
plt.ylim(0, 1)
graph = plt.scatter([], [])

def animate(i):
    graph.set_offsets(np.vstack((x[:i+1], y[:i+1])).T)
    graph.set_sizes(size[:i+1])
    graph.set_facecolors(colors[:i+1])
    return graph

ani = FuncAnimation(fig, animate, repeat=False, interval=200)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import matplotlib.animation as animation

fig, ax = plt.subplots()
t = np.linspace(0, 3, 40)
g = -9.81
v0 = 12
z = g * t**2 / 2 + v0 * t

v02 = 5
z2 = g * t**2 / 2 + v02 * t

scat = ax.scatter(t[0], z[0], c="b", s=5, label=f'v0 = {v0} m/s')
line2 = ax.plot(t[0], z2[0], label=f'v0 = {v02} m/s')[0]
ax.set(xlim=(0, 3), ylim=(-4, 10), xlabel='Time [s]', ylabel='Z [m]')
ax.legend()


def update(frame):
    # for each frame, update the data stored on each artist.
    x = t[:frame]
    y = z[:frame]
    # update the scatter plot:
    data = np.stack([x, y]).T
    scat.set_offsets(data)
    # update the line plot:
    line2.set_xdata(t[:frame])
    line2.set_ydata(z2[:frame])
    return (scat, line2)


ani = animation.FuncAnimation(fig=fig, func=update, frames=40, interval=30)
plt.show()